# PhiState A100 coherence journal

Select **Runtime → Change runtime type → A100 GPU**, then run the cell once. Authorize the Drive account that holds `MyDrive/Quantonium/PhiState-L4` when Colab prompts you. The run starts from the saved SFT-4,000 weights, checks the training and evaluation setup, and compares two continuations under the same budget. The scored task families are arithmetic, text transformations, questions grounded in supplied facts, and short Python functions. Separate broad prompts and paired multi-turn conversations check whether the answer changes when the user supplies different facts. If the candidate falls short, a bounded comparison with a conventional dense model runs automatically to help explain why. Prior checkpoints and the current native chat stay in their existing folders.

**After a disconnect:** reconnect to an A100 in Colab and run this same cell. Progress is saved in `MyDrive/Quantonium/PhiState-L4/whitespace_repair_v1/phistate_a100_coherence_v2/`. If an interrupted runtime still holds the run lock, confirm that the previous session has ended before enabling **RECOVER_DISCONNECTED**. The cell streams progress and prints the Drive path of its report. **status = complete** means the experiment finished. **outcome** records whether the candidate passed. Native export requires passing the broad and paired quality gates, at least half the changed-history pairs in both development and sealed dialogue evaluation, and successful C++ parity. The verified candidate is exported separately under `qualified_native/`. A stopped or failed run keeps the last committed checkpoints. Interrupt the cell to request a controlled stop.

**Automatic runtime release is off by default.** If PhiState has a dedicated A100, tick **RELEASE_GPU_AFTER_FINISH** before running to end the allocation after the completed report is saved. Releasing the runtime ends every job on that same runtime, including Wan. The report is saved as `phistate_a100_coherence_v2/report.json`.


In [ ]:
#@title Run PhiState on A100
RELEASE_GPU_AFTER_FINISH = False #@param {type:"boolean"}
RECOVER_DISCONNECTED = False #@param {type:"boolean"}

import importlib.util
import json
import os
import shutil
import signal
import subprocess
import sys
import tempfile
from pathlib import Path

import torch
from google.colab import drive

PIN = "73edfa1a704e8d198269351ebe597e5ff8fed8ed"
REPO = "https://github.com/Anharmoniclabs/quantoniumoslinux.git"
LAB = Path("runtime/experiments/phi_runtime_lab")
MOUNT = Path("/content/drive")
ROOT = MOUNT / "MyDrive/Quantonium/PhiState-L4/whitespace_repair_v1"
REPORT = ROOT / "phistate_a100_coherence_v2/report.json"

def require_a100():
    if not torch.cuda.is_available():
        raise RuntimeError("No CUDA GPU is attached. Select A100 under Runtime → Change runtime type, reconnect, and rerun.")
    gpu = torch.cuda.get_device_properties(0)
    if "A100" not in gpu.name.upper():
        raise RuntimeError(f"Attached GPU is {gpu.name}. This journal requires A100. Change runtime type, reconnect, and rerun.")
    if not hasattr(torch.optim, "Adafactor"):
        raise RuntimeError(f"PyTorch {torch.__version__} lacks Adafactor. Select the current Colab runtime version.")
    return gpu

def checkout(stage):
    def git(*args):
        return subprocess.check_output(["git", "-C", str(stage), *args], text=True).strip()
    git("init", "-q")
    git("remote", "add", "origin", REPO)
    git("config", "remote.origin.promisor", "true")
    git("config", "remote.origin.partialclonefilter", "blob:none")
    git("sparse-checkout", "set", "--no-cone", "/runtime/experiments/phi_runtime_lab/", "/runtime/src/rftmw_native/")
    git("fetch", "--quiet", "--depth=1", "--filter=blob:none", "origin", PIN)
    git("checkout", "--detach", "--quiet", "FETCH_HEAD")
    if git("rev-parse", "HEAD") != PIN or not (stage / LAB / "a100_coherence_journal.py").is_file():
        raise RuntimeError("Git checkout failed source verification.")

def foreground(command, directory):
    env = dict(os.environ, PYTHONDONTWRITEBYTECODE="1", PYTHONUNBUFFERED="1",
               PHISTATE_JOURNAL_COMMIT=PIN, TOKENIZERS_PARALLELISM="false")
    interrupted = False
    with subprocess.Popen(command, cwd=directory, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace", bufsize=1, start_new_session=True) as child:
        with child.stdout:
            while True:
                try:
                    for line in child.stdout:
                        print(line, end="", flush=True)
                    code = child.wait()
                    break
                except KeyboardInterrupt:
                    if not interrupted:
                        interrupted = True
                        print("Interrupt received. Waiting for the worker to stop and flush its current checkpoint.", flush=True)
                        try:
                            child.send_signal(signal.SIGINT)
                        except ProcessLookupError:
                            pass
    if code and not (interrupted and code in (-signal.SIGINT, 130)):
        raise RuntimeError(f"Worker exited with status {code}. Read the last worker message above; saved checkpoints remain on Drive.")
    return interrupted

def launch():
    gpu = require_a100()
    print(f"{gpu.name} | {gpu.total_memory/2**30:.1f} GiB | PyTorch {torch.__version__} | CUDA {torch.version.cuda}", flush=True)
    if not (MOUNT / "MyDrive").is_dir():
        try:
            drive.mount(str(MOUNT))
        except Exception as exc:
            raise RuntimeError("Drive authorization failed. Reconnect and grant access to the account that owns the PhiState folder, then rerun.") from exc
    source = ROOT / "coherence_execution_v1/source.pt"
    if not source.is_file():
        raise FileNotFoundError(f"The frozen SFT-4,000 checkpoint is missing from the mounted Drive: {source}")
    missing = [name for name in ("sentencepiece", "numpy") if importlib.util.find_spec(name) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
    if not shutil.which("g++"):
        subprocess.run(["apt-get", "-qq", "update"], check=True)
        subprocess.run(["apt-get", "-qq", "install", "-y", "g++"], check=True)
    print(f"Verified hardware | code commit {PIN} | output {REPORT.parent}", flush=True)
    with tempfile.TemporaryDirectory(prefix="phistate_a100_", dir="/content") as tmp:
        stage = Path(tmp)
        checkout(stage)
        lab = stage / LAB
        args = [sys.executable, "-u", str(lab / "a100_coherence_journal.py"), "--root", str(ROOT)]
        if RECOVER_DISCONNECTED:
            args.append("--recover-disconnected")
        interrupted = foreground(args, lab)
    if interrupted:
        print(f"Worker stopped. Drive output: {REPORT.parent}", flush=True)
        return
    if not REPORT.is_file():
        print(f"No final report was written. Read the worker's reason above. Run files: {REPORT.parent}", flush=True)
        return
    record = json.loads(REPORT.read_text(encoding="utf-8"))
    print(f"Drive report: {REPORT}", flush=True)
    for key in ("status", "decision", "native_export_eligible", "outcome", "conclusion", "next_action"):
        if key in record:
            print(f"{key}: {json.dumps(record[key], ensure_ascii=False)[:1200]}", flush=True)
    if RELEASE_GPU_AFTER_FINISH and record.get("status") == "complete":
        print("Report read back from Drive. Releasing this Colab runtime.", flush=True)
        try:
            from google.colab import runtime
            runtime.unassign()
        except Exception as exc:
            print(f"Use Runtime → Disconnect and delete runtime to release the GPU ({exc}).", flush=True)

launch()
